In [16]:
# import i wczytanie danych
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
from xgboost import XGBRegressor
import lightgbm as lgb

from collections import Counter

import joblib
import os

df = pd.read_csv(
    "english_bg3_reviews_with_sentiment_best_model.csv",
    sep=";",
    encoding="utf-8"
)
df.head()



,review_id,review,voted_up,author_steamid,date_str,review_length,review_word_count,clean_review,is_spam_label,num_emojis,low_quality,is_spam,tokens,sent_flair,review_category,future_expectations,has_expectation
0,209490463,The greatest game of all time. It simply is. \r\n,True,76561198842852247,2025-11-17 20:17:24,43,9,the greatest game of all time it simply is,0,0,0,0,"['the', 'greatest', 'game', 'all', 'time', 'si...",1,Unspecified,0,False
1,209488275,The dude trying to bite me at night was messed...,True,76561198335095296,2025-11-17 19:45:37,55,12,the dude trying to bite me at night was messed...,0,0,0,0,"['the', 'dude', 'trying', 'bite', 'night', 'wa...",0,Unspecified,0,False
2,209488128,"This is probably a great solo game, but my hop...",False,76561198031957731,2025-11-17 19:43:28,131,29,this is probably a great solo game but my hope...,0,0,0,0,"['this', 'probably', 'great', 'solo', 'game', ...",0,"balans, mechanika, soundtrack",1,True
3,209487979,One of the best RPG's of the last decade,True,76561198140742800,2025-11-17 19:41:16,39,9,one of the best rpgs of the last decade,0,0,0,0,"['one', 'the', 'best', 'rpgs', 'the', 'last', ...",1,Unspecified,0,False
4,209482559,Definitely one of the best games I've played i...,True,76561198024036703,2025-11-17 18:19:29,61,12,definitely one of the best games ive played in...,0,0,0,0,"['definitely', 'one', 'the', 'best', 'games', ...",1,Unspecified,0,False


In [17]:
df.shape[0]

53048

In [18]:
df.head()

,review_id,review,voted_up,author_steamid,date_str,review_length,review_word_count,clean_review,is_spam_label,num_emojis,low_quality,is_spam,tokens,sent_flair,review_category,future_expectations,has_expectation
0,209490463,The greatest game of all time. It simply is. \r\n,True,76561198842852247,2025-11-17 20:17:24,43,9,the greatest game of all time it simply is,0,0,0,0,"['the', 'greatest', 'game', 'all', 'time', 'si...",1,Unspecified,0,False
1,209488275,The dude trying to bite me at night was messed...,True,76561198335095296,2025-11-17 19:45:37,55,12,the dude trying to bite me at night was messed...,0,0,0,0,"['the', 'dude', 'trying', 'bite', 'night', 'wa...",0,Unspecified,0,False
2,209488128,"This is probably a great solo game, but my hop...",False,76561198031957731,2025-11-17 19:43:28,131,29,this is probably a great solo game but my hope...,0,0,0,0,"['this', 'probably', 'great', 'solo', 'game', ...",0,"balans, mechanika, soundtrack",1,True
3,209487979,One of the best RPG's of the last decade,True,76561198140742800,2025-11-17 19:41:16,39,9,one of the best rpgs of the last decade,0,0,0,0,"['one', 'the', 'best', 'rpgs', 'the', 'last', ...",1,Unspecified,0,False
4,209482559,Definitely one of the best games I've played i...,True,76561198024036703,2025-11-17 18:19:29,61,12,definitely one of the best games ive played in...,0,0,0,0,"['definitely', 'one', 'the', 'best', 'games', ...",1,Unspecified,0,False


In [19]:
# przygotowanie danych do predykcji

# sortowanie po dacie
df["date_str"] = pd.to_datetime(df["date_str"])
df = df.sort_values("date_str")

# true/false zamieniono na wartości int, żeby zmniejszyć szum informacyjny
df["voted_up"] = df["voted_up"].astype(int)

df.head()


,review_id,review,voted_up,author_steamid,date_str,review_length,review_word_count,clean_review,is_spam_label,num_emojis,low_quality,is_spam,tokens,sent_flair,review_category,future_expectations,has_expectation
53047,161353628,I've completed the game only one time through ...,1,76561198204577473,2024-03-24 14:59:22,277,54,ive completed the game only one time through a...,0,0,0,0,"['ive', 'completed', 'the', 'game', 'only', 'o...",1,"balans, soundtrack",0,False
53046,161354033,First playthrough was rough and i couldn't loo...,1,76561197968569873,2024-03-24 15:04:26,202,36,first playthrough was rough and i couldnt look...,0,0,0,0,"['first', 'playthrough', 'was', 'rough', 'and'...",1,balans,0,False
53045,161355385,350 hours played\r\n300 enemies (at least!) do...,1,76561199558514998,2024-03-24 15:20:26,800,156,hours played enemies at least doomed from the...,0,0,0,0,"['hours', 'played', 'enemies', 'least', 'doome...",0,"balans, fabuła, grafika",0,True
53044,161356090,pretty pretty pretty good. It took me about 40...,1,76561199028410237,2024-03-24 15:28:41,82,16,pretty pretty pretty good it took me about hou...,0,0,0,0,"['pretty', 'pretty', 'pretty', 'good', 'took',...",1,"grafika, mechanika",0,False
53043,161356717,I enjoy playing the game with a group of frien...,1,76561199581034929,2024-03-24 15:36:37,196,40,i enjoy playing the game with a group of frien...,0,0,0,0,"['enjoy', 'playing', 'the', 'game', 'with', 'g...",0,"balans, postacie",0,False


In [20]:
# date_str do datetime
df["date_str"] = pd.to_datetime(df["date_str"])

# wspólna grupa po dacie, agregacja per dzień
g = df.groupby(df["date_str"].dt.date)

# średni sentyment dzienny
daily_sentiment = g["sent_flair"].mean().to_frame(name="sentiment")
daily_sentiment.index = pd.to_datetime(daily_sentiment.index)
daily_sentiment.index.name = "date"

# liczba recenzji dziennie
daily_count = g["sent_flair"].count()
daily_count.index = pd.to_datetime(daily_count.index)
daily_count.index.name = "date"

daily_sentiment["count"] = daily_count

# skalowanie logarytmiczne liczby recenzji
daily_sentiment["count_log"] = np.log1p(daily_sentiment["count"])

daily_sentiment.head()


,sentiment,count,count_log
date,,,
2024-03-24,0.691176,68,4.234107
2024-03-25,0.798883,179,5.192957
2024-03-26,0.766234,154,5.043425
2024-03-27,0.674847,163,5.099866
2024-03-28,0.785047,107,4.682131


In [ ]:
# cechy czasowe i lags dla modeli 

# prosty licznik czasu
daily_sentiment["t"] = np.arange(len(daily_sentiment))

# cechy kalendarzowe
daily_sentiment["day_of_week"]   = daily_sentiment.index.dayofweek
daily_sentiment["week_of_year"]  = daily_sentiment.index.isocalendar().week.astype(int)
daily_sentiment["month"]         = daily_sentiment.index.month

# lags i średnia krocząca
daily_sentiment["lag1"]     = daily_sentiment["sentiment"].shift(1)
daily_sentiment["lag7"]     = daily_sentiment["sentiment"].shift(7)
daily_sentiment["rolling7"] = daily_sentiment["sentiment"].rolling(7).mean()

# usunięcie braków danych
daily_sentiment = daily_sentiment.dropna()
daily_sentiment.head()


,sentiment,count,count_log,t,day_of_week,week_of_year,month,lag1,lag7,rolling7
date,,,,,,,,,,
2024-03-31,0.797203,143,4.969813,7,6,13,3,0.762238,0.691176,0.773074
2024-04-01,0.831169,154,5.043425,8,0,14,4,0.797203,0.798883,0.777686
2024-04-02,0.743363,113,4.736198,9,1,14,4,0.831169,0.766234,0.774419
2024-04-03,0.848485,99,4.605170,10,2,14,4,0.743363,0.674847,0.799224
2024-04-04,0.798319,119,4.787492,11,3,14,4,0.848485,0.785047,0.801121


In [22]:
# train/test

FEATURES = ["t", "day_of_week", "week_of_year", "month", "lag1", "lag7", "rolling7", "count_log"]

X = daily_sentiment[FEATURES]
y = daily_sentiment["sentiment"]

train_size = int(len(daily_sentiment) * 0.8)

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# dodatkowo seria do Holt-Winters i Propheta
train = daily_sentiment.iloc[:train_size]
test  = daily_sentiment.iloc[train_size:]

len(train), len(test)


(477, 120)

In [23]:
# porównywanie 4 modeli - Prophet, xgboost, lightgbm, exponential smoothing
results = {}
predictions = {}


In [24]:
# Exponential Smoothing Holt-Winters
hw_model = ExponentialSmoothing(
    train["sentiment"],
    trend="add",
    seasonal=None,
    initialization_method="estimated"
).fit()

hw_pred = hw_model.forecast(len(test))

results["holt_winters"] = {
    "rmse": np.sqrt(mean_squared_error(test["sentiment"], hw_pred)),
    "mae":  mean_absolute_error(test["sentiment"], hw_pred)
}
predictions["holt_winters"] = hw_pred


In [25]:
# Prophet
prophet_df = daily_sentiment[["sentiment"]].copy()
prophet_df = prophet_df.reset_index().rename(columns={
    "date": "ds",
    "sentiment": "y"
})

prophet_train = prophet_df.iloc[:train_size]

prophet_model = Prophet()
prophet_model.fit(prophet_train)

prophet_future  = prophet_model.make_future_dataframe(periods=len(test), freq="D")
prophet_forecast = prophet_model.predict(prophet_future)

prophet_pred = prophet_forecast["yhat"].iloc[-len(test):].values

results["prophet"] = {
    "rmse": np.sqrt(mean_squared_error(test["sentiment"].values, prophet_pred)),
    "mae":  mean_absolute_error(test["sentiment"].values, prophet_pred)
}
predictions["prophet"] = prophet_pred


06:39:52 - cmdstanpy - INFO - Chain [1] start processing
06:39:52 - cmdstanpy - INFO - Chain [1] done processing


In [26]:
# xgboost
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

results["xgboost"] = {
    "rmse": np.sqrt(mean_squared_error(y_test, xgb_pred)),
    "mae":  mean_absolute_error(y_test, xgb_pred)
}
predictions["xgboost"] = xgb_pred


In [27]:
# lightgbm
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)

results["lightgbm"] = {
    "rmse": np.sqrt(mean_squared_error(y_test, lgb_pred)),
    "mae":  mean_absolute_error(y_test, lgb_pred)
}
predictions["lightgbm"] = lgb_pred


In [28]:
# best_model 
print("PORÓWNANIE:")
for model_name, metric in results.items():
    print(
        f"{model_name:12s} | RMSE: {metric['rmse']:.4f} | MAE: {metric['mae']:.4f}"
    )

best_model = min(results, key=lambda m: results[m]["rmse"])
print("\nNajlepszy model to ===", best_model)


PORÓWNANIE:
holt_winters | RMSE: 0.0601 | MAE: 0.0481
prophet      | RMSE: 0.0590 | MAE: 0.0478
xgboost      | RMSE: 0.0563 | MAE: 0.0438
lightgbm     | RMSE: 0.0606 | MAE: 0.0470

Najlepszy model to === xgboost


In [29]:
FUTURE_DAYS = 120

last_date = daily_sentiment.index.max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1),
                             periods=FUTURE_DAYS,
                             freq="D")

# cechy dla przyszłości
future_features = pd.DataFrame(index=future_dates)
future_features["t"] = np.arange(
    daily_sentiment["t"].max() + 1,
    daily_sentiment["t"].max() + 1 + FUTURE_DAYS
)
future_features["day_of_week"]  = future_features.index.dayofweek
future_features["week_of_year"] = future_features.index.isocalendar().week.astype(int)
future_features["month"]        = future_features.index.month

# dla lagów używamy ostatnich znanych wartości zeby zrobic proste przyblizenie
last_row = daily_sentiment.iloc[-1]
future_features["lag1"]     = last_row["lag1"]
future_features["lag7"]     = last_row["lag7"]
future_features["rolling7"] = last_row["rolling7"]
future_features["count_log"] = last_row["count_log"]


# predykcja zależnie best_model
if best_model == "xgboost":
    future_pred = xgb_model.predict(future_features[FEATURES])
elif best_model == "lightgbm":
    future_pred = lgb_model.predict(future_features[FEATURES])
elif best_model == "holt_winters":
    future_pred = hw_model.forecast(FUTURE_DAYS)
elif best_model == "prophet":
    prophet_future_all = prophet_model.make_future_dataframe(periods=FUTURE_DAYS, freq="D")
    prophet_forecast_all = prophet_model.predict(prophet_future_all)
    future_part = prophet_forecast_all.tail(FUTURE_DAYS)
    future_pred = future_part["yhat"].values
    future_dates = future_part["ds"].values  # daty z propheta



In [30]:
# zapis najlepszego wytrenowanego modelu

model_path = 'models/sentiment_forecast_model.pkl'

if best_model == "prophet":
    joblib.dump(prophet_model, model_path)
elif best_model == "xgboost":
    joblib.dump(xgb_model, model_path)
elif best_model == "lightgbm":
    joblib.dump(lgb_model, model_path)
elif best_model == "holt_winters":
    joblib.dump(hw_model, model_path)

In [31]:
# to csv
future_df = pd.DataFrame({
    "date": future_dates,
    "predicted_sentiment": future_pred
})

future_df.to_csv("english_bg3_prediction.csv", index=False, sep=";", encoding="utf-8")
future_df.head()


,date,predicted_sentiment
0,2025-11-18,0.816156
1,2025-11-19,0.816804
2,2025-11-20,0.815698
3,2025-11-21,0.818576
4,2025-11-22,0.812158
